# AutoResearch — Colab train + register (Phase 4)

**Chain:**

`GPU → TinyStories → tokenizer → train → val_bpb → checkpoint → result JSON → reload → POST FastAPI → Supabase`

1. Runtime → **GPU**
2. Run training cells
3. Set `AUTORESEARCH_API_URL` to a URL that reaches your FastAPI (tunnel/public host)
4. Run registration cell → expect `REGISTER: PASS`

Smoke training budget: **30s**. Production default remains `TIME_BUDGET=300`.

In [ ]:
# STEP 1 — CUDA
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print("PASS STEP 1 CUDA:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# STEP 2 — clone / update repo
from pathlib import Path

REPO = "autoresearch-platform"
URL = "https://github.com/justSamarthings/autoresearch-platform.git"

if Path(REPO).exists():
    %cd {REPO}
    !git fetch origin
    !git checkout main
    !git pull --ff-only origin main
else:
    !git clone {URL}
    %cd {REPO}

print("PASS STEP 2 REPO:", Path.cwd(), "|", end=" ")
!git rev-parse --short HEAD

In [ ]:
# STEP 3 — training deps (Torch comes from Colab)
%cd training
!pip install -q -r requirements.txt
!pip install -q "kernels>=0.11.7" || true
print("PASS STEP 3 DEPS")

In [ ]:
# STEP 4 — TinyStories prepare + tokenizer
!python prepare.py

from pathlib import Path
import os

cache = Path(os.environ.get("AUTORESEARCH_CACHE", Path.home() / ".cache" / "autoresearch-platform"))
assert (cache / "data" / "train.parquet").exists()
assert (cache / "tokenizer" / "tokenizer.pkl").exists()
print("PASS STEP 4 TINYSTORIES+TOKENIZER:", cache)

In [ ]:
# STEP 5 — 30s smoke train
!AUTORESEARCH_TIME_BUDGET=30 AUTORESEARCH_NO_COMPILE=1 python train.py
print("PASS STEP 5 TRAIN finished")

In [ ]:
# STEP 6 — local artifacts + checkpoint reload
from pathlib import Path
import json
import subprocess
import sys

ckpt_dir = Path("artifacts/checkpoints")
res_dir = Path("artifacts/results")
ckpts = sorted(ckpt_dir.glob("*.pt"))
results = sorted(res_dir.glob("*.json"))
assert ckpts and results

ckpt = ckpts[-1]
result_path = results[-1]
result = json.loads(result_path.read_text())
assert result.get("status") == "ok"
assert result.get("val_bpb") is not None

print("PASS STEP 6a val_bpb:", result["val_bpb"])
print("PASS STEP 6b checkpoint:", ckpt)
print("PASS STEP 6c result JSON:", result_path)

proc = subprocess.run(
    [sys.executable, "../scripts/verify_checkpoint.py", str(ckpt)],
    check=False, capture_output=True, text=True,
)
print(proc.stdout)
assert proc.returncode == 0, proc.stderr
print("PASS STEP 6d checkpoint reload")
print("SMOKE CHAIN: PASS")

## STEP 7 — register result with FastAPI → Supabase

Your Colab runtime **cannot** reach `http://127.0.0.1:8000` on your laptop.

1. On your machine (repo root), start the API:
   `uvicorn backend.app.main:app --host 0.0.0.0 --port 8000`
2. Expose it with a tunnel (e.g. Cloudflare Tunnel / ngrok) **or** deploy the API.
3. Paste that public base URL below (no trailing path).

In [ ]:
# STEP 7 — register latest result JSON with the platform API
import os
from pathlib import Path

# REQUIRED: public/tunnel URL that reaches your FastAPI process
os.environ["AUTORESEARCH_API_URL"] = os.environ.get(
    "AUTORESEARCH_API_URL",
    "https://YOUR-TUNNEL-OR-HOST",  # <-- replace
)

assert not os.environ["AUTORESEARCH_API_URL"].endswith("YOUR-TUNNEL-OR-HOST"), (
    "Set AUTORESEARCH_API_URL to your reachable FastAPI base URL"
)

# Ensure we are in training/ (artifact paths) then call registrar from repo scripts/
if Path("artifacts/results").exists():
    root = Path("..").resolve()
elif Path("training/artifacts/results").exists():
    root = Path(".").resolve()
else:
    raise SystemExit("Could not find artifacts/results — run training steps first")

%cd {root}
!python scripts/register_experiment.py --latest --verify
print("REGISTER: PASS")

### Optional: full 5-minute experiment

```python
%cd training
!python train.py
%cd ..
!python scripts/register_experiment.py --latest --verify
```